# 02 · Weather analytics

**Owner: Weather Analytics.** Event rates use the number of *valid observations* as the denominator. This sample uses relaxed demo thresholds because only 20 observations/station/year exist.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LivestockWeatherNotebook").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from livestock_weather.settings import load_settings
cfg = load_settings(ROOT / "config" / "demo.json")


In [ ]:
from livestock_weather.analytics import state_year_metrics, state_summary, state_seasonal_metrics
clean = ROOT / "data" / "demo_processed" / "clean_daily.parquet"
if not clean.exists():
    raise FileNotFoundError("Run 01_data_processing.ipynb first")
daily = spark.read.parquet(str(clean))
annual = state_year_metrics(daily, cfg)
summary = state_summary(annual)
seasonal = state_seasonal_metrics(daily, cfg)
annual.orderBy("state", "year").show(20, False)
# Example Spark SQL on the cleaned Parquet DataFrame (exploratory, not risk scoring).
daily.createOrReplaceTempView("clean_daily")
spark.sql("""SELECT state, year, COUNT(DISTINCT station_id) AS observed_stations,
                    ROUND(AVG(tmax_c), 1) AS mean_observed_tmax_c
             FROM clean_daily GROUP BY state, year ORDER BY state, year""").show()

In [ ]:
seasonal.orderBy("state", "season").show(16, False)
for name, frame in (("state_annual", annual), ("state_summary", summary), ("state_seasonal", seasonal)):
    frame.write.mode("overwrite").parquet(str(ROOT / "data" / "demo_processed" / (name + ".parquet")))